<a href="https://colab.research.google.com/github/laercioives/prevendo-faltas-em-consutas-medicas/blob/main/Predict_NoShow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import pandas as pd
import kagglehub

# 1. Baixa o dataset (retorna o caminho do diretório)
path = kagglehub.dataset_download("joniarroba/noshowappointments")

# 2. Lista os arquivos dentro da pasta para encontrar o nome do CSV
arquivos = os.listdir(path)
print("Arquivos encontrados na pasta:", arquivos)

# 3. Pega o caminho completo do arquivo CSV (normalmente 'KaggleV2-May-2016.csv')
caminho_csv = os.path.join(path, arquivos[0])

# 4. Carrega o arquivo em um DataFrame do Pandas
df = pd.read_csv(caminho_csv)

# 5. Visualiza as primeiras linhas
df.head()



100%|██████████| 2.40M/2.40M [00:00<00:00, 126MB/s]

Extracting files...
Arquivos encontrados na pasta: ['KaggleV2-May-2016.csv']


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


Treinando os modelos de predição

In [7]:
# df['Neighbourhood'].unique()
df

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110522,2.572134e+12,5651768,F,2016-05-03T09:15:35Z,2016-06-07T00:00:00Z,56,MARIA ORTIZ,0,0,0,0,0,1,No
110523,3.596266e+12,5650093,F,2016-05-03T07:27:33Z,2016-06-07T00:00:00Z,51,MARIA ORTIZ,0,0,0,0,0,1,No
110524,1.557663e+13,5630692,F,2016-04-27T16:03:52Z,2016-06-07T00:00:00Z,21,MARIA ORTIZ,0,0,0,0,0,1,No
110525,9.213493e+13,5630323,F,2016-04-27T15:09:23Z,2016-06-07T00:00:00Z,38,MARIA ORTIZ,0,0,0,0,0,1,No


In [8]:
import pandas as pd
import numpy as np

# Supondo que seu DataFrame inicial se chame 'df'
# df = pd.read_csv(caminho_csv)

# -------------------------------------------------------------------------
# 1. Remoção dos IDs (PatientId, AppointmentID)
# -------------------------------------------------------------------------
df = df.drop(columns=['PatientId', 'AppointmentID'])

# -------------------------------------------------------------------------
# 2. Conversão da variável 'Gender' (0 para 'M' e 1 para 'F')
# -------------------------------------------------------------------------
df['Gender'] = df['Gender'].map({'M': 0, 'F': 1})

# -------------------------------------------------------------------------
# 3. Engenharia de Recursos a partir das Datas
# -------------------------------------------------------------------------
# Convertendo para o formato datetime do Pandas
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'])
df['ScheduledDay'] = pd.to_datetime(df['ScheduledDay'])

# Extração a partir do AppointmentDay
df['App_DayOfWeek'] = df['AppointmentDay'].dt.dayofweek  # 0=Segunda, 6=Domingo
df['App_DayOfMonth'] = df['AppointmentDay'].dt.day
df['App_Month'] = df['AppointmentDay'].dt.month
df['App_IsWeekend'] = df['App_DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)

# Extração do turno a partir do horário em ScheduledDay
def definir_turno(hora):
    if 5 <= hora < 12:
        return 'matutino'
    elif 12 <= hora < 18:
        return 'vespertino'
    else:
        return 'noturno'

df['Scheduled_Hour'] = df['ScheduledDay'].dt.hour
df['Scheduled_Period'] = df['Scheduled_Hour'].apply(definir_turno)

# Aplica One-Hot Encoding no turno criado (matutino, vespertino, noturno)
df = pd.get_dummies(df, columns=['Scheduled_Period'], drop_first=False, dtype=int)

# -------------------------------------------------------------------------
# 4. Remoção de 'Neighbourhood' e dos campos principais de Data
# -------------------------------------------------------------------------
df = df.drop(columns=['Neighbourhood', 'AppointmentDay', 'ScheduledDay', 'Scheduled_Hour'])

# -------------------------------------------------------------------------
# 5. Mapeamento da variável Alvo 'No-show' (0 para 'No' e 1 para 'Yes')
# -------------------------------------------------------------------------
# Garantindo tratamento de variações de maiúsculas/minúsculas
df['No-show'] = df['No-show'].astype(str).str.strip().str.capitalize()
df['No-show'] = df['No-show'].map({'No': 0, 'Yes': 1})

# -------------------------------------------------------------------------
# 6. Tratamento de Outliers / Inconsistências e Dados Ausentes
# -------------------------------------------------------------------------
# Correção de idades negativas (inconsistência conhecida deste dataset)
df = df[df['Age'] >= 0]

# Tratamento de dados ausentes (caso existam no carregamento)
# Para variáveis numéricas: preenche com a mediana
num_cols = df.select_dtypes(include=[np.number]).columns
for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

# -------------------------------------------------------------------------
# 7. Verificação Final do DataFrame
# -------------------------------------------------------------------------
print("--- PRÉ-PROCESSAMENTO CONCLUÍDO ---")
print(f"Dimensões finais: {df.shape}")
print(f"Valores nulos restantes: {df.isnull().sum().sum()}")
print("\nPrimeiras linhas do DataFrame pré-processado:")
df.head()



--- PRÉ-PROCESSAMENTO CONCLUÍDO ---
Dimensões finais: (110526, 16)
Valores nulos restantes: 0

Primeiras linhas do DataFrame pré-processado:


,Gender,Age,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show,App_DayOfWeek,App_DayOfMonth,App_Month,App_IsWeekend,Scheduled_Period_matutino,Scheduled_Period_noturno,Scheduled_Period_vespertino
0,1,62,0,1,0,0,0,0,0,4,29,4,0,0,1,0
1,0,56,0,0,0,0,0,0,0,4,29,4,0,0,0,1
2,1,62,0,0,0,0,0,0,0,4,29,4,0,0,0,1
3,1,8,0,0,0,0,0,0,0,4,29,4,0,0,0,1
4,1,56,0,1,1,0,0,0,0,4,29,4,0,0,0,1


In [9]:
# 1. Definindo a variável alvo (y)
y = df['No-show']

# 2. Definindo a matriz de atributos/variáveis preditoras (X)
X = df.drop(columns=['No-show'])

Realizando a Predição com Random Forests


In [11]:
import numpy as np
import pandas as pd

# Modelo e Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

# Validação e Métricas
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate
from sklearn.metrics import make_scorer, recall_score, accuracy_score, confusion_matrix

# -------------------------------------------------------------------------
# 1. Definição das Métricas Personalizadas (Especificidade e G-Mean)
# -------------------------------------------------------------------------
def specificity_score(y_true, y_pred):
    """Calcula a Especificidade: TN / (TN + FP)"""
    cm = confusion_matrix(y_true, y_pred)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        return tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return 0.0

def gmean_score(y_true, y_pred):
    """Calcula a Média Geométrica (G-Mean) entre Sensibilidade e Especificidade"""
    sens = recall_score(y_true, y_pred, zero_division=0)
    spec = specificity_score(y_true, y_pred)
    return np.sqrt(sens * spec)

# Dicionário com todas as métricas solicitadas
scoring = {
    'auc_roc': 'roc_auc',
    'sensibilidade': make_scorer(recall_score, zero_division=0),
    'especificidade': make_scorer(specificity_score),
    'acuracia': make_scorer(accuracy_score),
    'gmean': make_scorer(gmean_score)
}

# -------------------------------------------------------------------------
# 2. Configuração do Pipeline e da Grade de Hiperparâmetros
# -------------------------------------------------------------------------
pipeline = Pipeline([
    ('model', RandomForestClassifier(random_state=42))
])

param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [3, 5, 7, 10],
    'model__min_samples_split': [2, 5, 10, 15],
    'model__class_weight': ['balanced']
}

# -------------------------------------------------------------------------
# 3. Estruturação da Validação Cruzada Aninhada (Nested CV 5x5)
# -------------------------------------------------------------------------
cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Grid Search configurado para otimizar ROC-AUC no loop interno
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv_inner,
    scoring='roc_auc',
    n_jobs=10
)

print("Iniciando o treinamento e avaliação com Validação Cruzada Aninhada (Nested CV)...")

# Avaliação externa (5-fold cross-validation)
cv_results = cross_validate(
    grid_search,
    X, y,
    cv=cv_outer,
    scoring=scoring,
    n_jobs=10,
    return_train_score=False
)

# -------------------------------------------------------------------------
# 4. Compilação e Exibição dos Resultados
# -------------------------------------------------------------------------
resultados = {
    'Métrica': ['AUC ROC', 'Sensibilidade', 'Especificidade', 'Acurácia', 'G-Mean'],
    'Média': [
        np.mean(cv_results['test_auc_roc']),
        np.mean(cv_results['test_sensibilidade']),
        np.mean(cv_results['test_especificidade']),
        np.mean(cv_results['test_acuracia']),
        np.mean(cv_results['test_gmean'])
    ],
    'Desvio Padrão': [
        np.std(cv_results['test_auc_roc']),.n.m.m.m.m.n. ,,,,,,,z
        np.std(cv_results['test_sensibilidade']),
        np.std(cv_results['test_especificidade']),
        np.std(cv_results['test_acuracia']),
        np.std(cv_results['test_gmean'])
    ]
}

df_resultados = pd.DataFrame(resultados)

print("\n--- RESULTADOS FINAIS DO RANDOM FOREST (OUTER CV) ---")
display(df_resultados.round(4))

Iniciando o treinamento e avaliação com Validação Cruzada Aninhada (Nested CV)...

--- RESULTADOS FINAIS DO RANDOM FOREST (OUTER CV) ---


,Métrica,Média,Desvio Padrão
0,AUC ROC,0.6321,0.0031
1,Sensibilidade,0.5444,0.0104
2,Especificidade,0.6477,0.0088
3,Acurácia,0.6269,0.0049
4,G-Mean,0.5937,0.0017
